# Stock Return Prediction

# Data Collection

# Problem Framing and Use Case

This notebook focuses on predicting **stock returns** for Apple (AAPL) using various machine learning models, including Ridge Regression, XGBoost, and Long Short-Term Memory (LSTM) networks. The primary goal is to forecast the daily logarithmic return of the stock.

**Use Case:**

The ability to accurately predict stock returns, especially the direction of movement, is invaluable in quantitative finance and algorithmic trading. Such predictions can be used for:

*   **Investment Strategies:** Informing buy/sell decisions for individual investors or institutional funds.
*   **Risk Management:** Assessing potential future volatility and adjusting portfolio risk accordingly.
*   **Portfolio Optimization:** Constructing portfolios that aim to maximize returns for a given level of risk.
*   **Algorithmic Trading:** Developing automated trading systems that execute trades based on predicted movements.

By comparing different models, we aim to understand their effectiveness in capturing the complex, non-linear dynamics of stock markets and identify which approach provides the most reliable predictive signals, particularly concerning the direction of price changes.

First, we import the necessary libraries:
-   `numpy` for numerical operations.
-   `pandas` for data manipulation and analysis.
-   `matplotlib.pyplot` for data visualization.
-   `yfinance` to download historical stock data.
-   `sklearn.preprocessing.StandardScaler` for feature scaling.

In [ ]:
import numpy as np # Import numpy for numerical operations
import pandas as pd # Import pandas for data manipulation and analysis
import matplotlib.pyplot as plt # Import matplotlib for data visualization
import yfinance as yf # Import yfinance to download historical stock data
from sklearn.preprocessing import StandardScaler # Import StandardScaler for feature scaling

Next, we download the historical stock data for Apple (AAPL) from 2015 to 2024. We then select the 'Open', 'High', 'Low', and 'Volume' columns. After that, we calculate the daily logarithmic return and create a 'target' variable by shifting the log return by one day. Finally, we remove any rows with missing values that result from these calculations.

In [ ]:
# Download Apple (AAPL) stock data with auto-adjustment
df = yf.download("AAPL", start="2015-01-01", end="2024-12-31", auto_adjust=True)

# Select specific columns including 'Close'
df = df[['Open', 'High', 'Low', 'Close', 'Volume']]

# Flatten multi-level column names to single strings
# Example: ('Open', 'AAPL') becomes 'Open'
df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns.values]

# Compute log return and target
df['log_return'] = np.log(df['Close'] / df['Close'].shift(1)) # Calculate daily logarithmic return
# Shift the log return to create the target variable
df['target'] = df['log_return'].shift(-1)

# Remove rows with NaN values resulting from log return or target calculation
df.dropna(inplace=True)
# Display the first few rows of the DataFrame
df.head()

In [ ]:
# Display information about the DataFrame, including data types and non-null values
df.info()


# Feature Engineering

This `info()` output provides a summary of the DataFrame, including the number of entries, the number of columns, the data type of each column, and the count of non-null values. It's a quick way to check for potential data type issues or missing data at a glance.

The `describe()` method provides a statistical summary of the numerical columns in the DataFrame, including count, mean, standard deviation, min, max, and quartile values. This helps us understand the distribution and potential outliers in our data.

In [ ]:
# Generate descriptive statistics for numerical columns
df.describe()

Now, we will create lag features for the `log_return` column. Lag features are previous values of a time series, which can be useful as predictors for future values. We will create lags from 1 to 5 days.

### Lag Features

**Reasoning:**
Lag features are past values of a time series that are used as predictor variables. For stock return prediction, previous days' returns can be highly indicative of current or future returns. By creating `log_return_lag1` through `log_return_lag5`, we introduce the historical context of daily logarithmic returns into our model.

The `df.dropna(inplace=True)` call is essential here because creating lag features inherently introduces `NaN` values at the beginning of the DataFrame (where there are no preceding values to shift from). Removing these rows ensures that our dataset contains only complete feature sets, which is required for most machine learning models.

In [ ]:
# Create lag features for log_return
for i in range(1, 6):
    df[f'log_return_lag{i}'] = df['log_return'].shift(i)

# Remove rows with NaN values resulting from lag feature creation
df.dropna(inplace=True)

# Display the first few rows of the DataFrame with new lag features
df.head()

Next, we'll compute rolling statistics for `log_return`. Rolling statistics (also known as moving averages or moving standard deviations) can help capture trends and volatility over specific periods, which can be valuable features for prediction models. We will create rolling mean and standard deviation for windows of 5 and 20 days.

### Rolling Statistics

**Reasoning:**
Rolling statistics, such as moving averages and moving standard deviations, are crucial for capturing trends and volatility over specified periods in time series data. They act as smoothed versions of the original data, helping to reduce noise and highlight underlying patterns.

-   **`rolling_mean_5`** and **`rolling_mean_20`**: These features represent the average `log_return` over the past 5 and 20 trading days, respectively. They help in identifying short-term and medium-term trends.
-   **`rolling_std_5`** and **`rolling_std_20`**: These features measure the volatility of `log_return` over the past 5 and 20 trading days. Higher standard deviations indicate greater price fluctuations.

Similar to lag features, calculating rolling statistics introduces `NaN` values at the beginning of the DataFrame, as there aren't enough preceding data points to fill the initial windows. Therefore, `df.dropna(inplace=True)` is used to remove these rows, ensuring all entries have valid rolling statistics.

In [ ]:
df['rolling_mean_5'] = df['log_return'].rolling(window=5).mean()
df['rolling_mean_20'] = df['log_return'].rolling(window=20).mean()
df['rolling_std_5'] = df['log_return'].rolling(window=5).std()
df['rolling_std_20'] = df['log_return'].rolling(window=20).std()

# Remove rows with NaN values resulting from rolling statistics calculation
df.dropna(inplace=True)

# Display the first few rows of the DataFrame with new rolling features
df.head()

Now, let's incorporate features based on trading volume. Volume can be a crucial indicator of market sentiment and momentum. We will create two new volume-related features:
1.  **`log_volume`**: The natural logarithm of the trading volume.
2.  **`log_volume_change`**: The daily change (logarithmic return) of the `log_volume`.

### Volume-Related Features

**Reasoning:**
Trading volume provides crucial insights into market activity and sentiment. By incorporating volume-based features, we can capture how strong a price movement is, which can be an important predictor of future price behavior.

-   **`log_volume`**: The natural logarithm of the trading volume helps to normalize the data. Raw volume can vary widely, and its logarithmic transformation can make the feature more stable and prevent large values from disproportionately influencing the model. It reflects the overall level of trading activity.
-   **`log_volume_change`**: The daily change in `log_volume` (calculated as `df['log_volume'].diff()`) indicates the rate of change in trading activity. Significant increases or decreases in volume can signal shifts in market interest, potential reversals, or continuation of trends.

As with other feature engineering steps, calculating `log_volume_change` introduces `NaN` values for the first entry. Therefore, `df.dropna(inplace=True)` is used to ensure all rows have complete data, which is essential for model training.

In [ ]:
# Calculate the natural logarithm of the volume
df['log_volume'] = np.log(df['Volume'])
# Calculate the daily change in log volume
df['log_volume_change'] = df['log_volume'].diff()


# Remove rows with NaN values resulting from these calculations
df.dropna(inplace=True)

# Display the first few rows of the DataFrame with the new volume features
df.head()

# Train/Test Split

## Select Features

### Subtask:
Identify and select the features (X) and the target variable (y) from the DataFrame. The target variable is 'target', and features will include 'log_return_lag1' through 'log_return_lag5', 'rolling_mean_5', 'rolling_mean_20', 'rolling_std_5', 'rolling_std_20' and 'log_volume_change'.


**Reasoning**:
I will create a list of feature names, then select these features to create the `X` DataFrame and select the 'target' column to create the `y` Series.



In [ ]:
features = [
    'log_return_lag1', 'log_return_lag2', 'log_return_lag3', 'log_return_lag4', 'log_return_lag5',
    'rolling_mean_5', 'rolling_mean_20', 'rolling_std_5', 'rolling_std_20',
    'log_volume', 'log_volume_change'
]

X = df[features] # Select the features
y = df['target'] # Select the target variable

print(f"Shape of X: {X.shape}") # Display the shape of the features DataFrame
print(f"Shape of y: {y.shape}") # Display the shape of the target Series


### Subtask:
Split the features (X) and target (y) DataFrames into training and testing sets using an 80/20 chronological split.


## Split Data Chronologically

# Task
Select the specified features and target variable from the DataFrame, then perform a chronological split of the dataset into training and testing sets.


**Reasoning**:
To perform the chronological split, I will first calculate the split index based on 80% of the data length. Then, I will use this index to divide the `X` and `y` DataFrames into `X_train`, `X_test`, `y_train`, and `y_test` respectively.



In [ ]:
# Calculate the split index for 80% training data
train_index = int(len(X) * 0.70)
val_index = int(len(X) * 0.85)

# Create the training set for features
X_train = X.iloc[:train_index]
# Create the validation set for features
X_val = X.iloc[train_index:val_index]
# Create the testing set for features
X_test = X.iloc[val_index:]

# Create the training set for the target variable
y_train = y.iloc[:train_index]
# Create the validation set for the target variable
y_val = y.iloc[train_index:val_index]
# Create the testing set for the target variable
y_test = y.iloc[val_index:]

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_val: {X_val.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_val: {y_val.shape}")
print(f"Shape of y_test: {y_test.shape}")

# Modeling

## Task
Perform feature scaling on the training, validation, and test datasets using `StandardScaler` to prevent data leakage. Specifically, fit the scaler on `X_train`, then transform `X_train`, `X_val`, and `X_test` to create `X_train_scaled`, `X_val_scaled`, and `X_test_scaled`. Finally, verify the scaling by displaying the descriptive statistics of `X_train_scaled`.

## Initialize and Fit Scaler

### Subtask:
Initialize a scaler and fit it exclusively on the training feature data (`X_train`). This prevents data leakage from the validation and test sets.


**Reasoning**:
I will instantiate the `StandardScaler`, and fit it on `X_train` to prepare the data for scaling.



In [ ]:
# Instantiate the StandardScaler
scaler = StandardScaler()

# Fit the scaler only on the training data to prevent data leakage
scaler.fit(X_train)

print("StandardScaler initialized and fitted on X_train.")

**Reasoning**:
Now that the `StandardScaler` has been fitted on `X_train`, I will use it to transform `X_train`, `X_val`, and `X_test` to create scaled versions of these datasets. This ensures that the scaling is consistent across all data splits and prevents data leakage.



In [ ]:
# Transform the training, validation, and test data using the fitted scaler
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert the scaled arrays back to DataFrames, preserving column names and index
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns, index=X_val.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("X_train, X_val, and X_test successfully scaled.")

## Data Verification and Split Confirmation

**Reasoning**:
This cell serves a crucial verification step after data preprocessing and splitting:

1.  **Log Return Distribution**: The histogram of `log_return` helps visually confirm its distribution, often expected to be roughly normal, which can influence model choice and interpretation.
2.  **Split Overlap Check**: Printing the date ranges for the training, validation, and test sets (`df.index[0].date()`, etc.) explicitly confirms that there is no temporal overlap between these sets. This is vital for time series data to ensure that the model is evaluated on future, unseen data and to prevent data leakage.
3.  **Shape Confirmation**: Displaying the shapes of `X_train_scaled`, `X_val_scaled`, and `X_test_scaled` (e.g., `(1741, 11)`) verifies that the scaling process has maintained the correct dimensions and that the data has been split as intended, confirming the number of samples and features in each set.

In [ ]:
# Log return distribution — should look roughly normal
df["log_return"].hist(bins=100, figsize=(8, 4))
plt.title("Log Return Distribution")
plt.show()

# Confirm no overlap between splits
print("Train:", df.index[0].date(), "→", df.index[train_index - 1].date())
print("Val:  ", df.index[train_index].date(), "→", df.index[val_index - 1].date())
print("Test: ", df.index[val_index].date(), "→", df.index[-1].date())

# Confirm shapes
print(X_train_scaled.shape, X_val_scaled.shape, X_test_scaled.shape)

## Ridge Regression Model

#### Introduction
Now that our data has been preprocessed and scaled, we will move on to the modeling phase. For this task, we will employ Ridge Regression as our baseline model to predict stock returns. Ridge Regression is a type of linear regression that includes an L2 regularization term, which helps to prevent overfitting by penalizing large coefficients.

#### Reasoning
Ridge Regression is a suitable choice for a baseline model in this context due to several factors:
1.  **Simplicity:** As a linear model, it provides a straightforward and interpretable starting point for predicting stock returns.
2.  **Effectiveness with Multicollinearity:** Stock market data often exhibits multicollinearity among features. Ridge Regression handles this by adding a penalty proportional to the square of the magnitude of the coefficients, which can stabilize the model and improve generalization when features are correlated.
3.  **Interpretability:** The coefficients of a Ridge Regression model can still provide insights into the relationship between our features and the target variable, making it easier to understand which factors influence stock returns.
4.  **Robustness:** It is less sensitive to outliers compared to ordinary least squares regression, which is beneficial given the inherent volatility of financial data.

### Task
Setup Ridge Regression as the baseline model.

### Instantiate and Fit Ridge Regression Model

### Subtask:
Initialize a Ridge Regression model with `alpha=1` and fit it using the `X_train_scaled` and `y_train` datasets.


**Reasoning**:
I need to import the `Ridge` model, instantiate it with `alpha=1`, and then fit it to the training data. This will be done in a code block.



In [ ]:
from sklearn.linear_model import Ridge

# Instantiate a Ridge Regression model with alpha=1
ridge_model = Ridge(alpha=1)

# Fit the model to the scaled training data
ridge_model.fit(X_train_scaled, y_train)

print("Ridge Regression model initialized and fitted.")

### Make Predictions

### Subtask:
Generate predictions on X_val_scaled and X_test_scaled using the fitted Ridge Regression model.


**Reasoning**:
I will use the fitted `ridge_model` to generate predictions on `X_val_scaled` and `X_test_scaled` and store them in `y_pred_val` and `y_pred_test`.



In [ ]:
y_pred_val = ridge_model.predict(X_val_scaled)
y_pred_test = ridge_model.predict(X_test_scaled)

print("Predictions generated for validation and test sets.")

## Define Evaluation Function

### Subtask:
Create a Python function that takes true values and predicted values as input, and calculates Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and Directional Accuracy. This function will be reusable for evaluating model performance.


**Reasoning**:
I need to import the necessary metrics from `sklearn.metrics` and define a function that calculates MAE, RMSE, and directional accuracy as specified in the instructions. This function will be placed in a new code cell.



In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate_model(y_true, y_pred):
    # Calculate Mean Absolute Error (MAE)
    mae = mean_absolute_error(y_true, y_pred)

    # Calculate Root Mean Squared Error (RMSE)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # Calculate Directional Accuracy
    # Compare the sign of true values with predicted values
    # Directional accuracy is the proportion of correct sign predictions
    directional_accuracy = np.mean(np.sign(y_true) == np.sign(y_pred))

    return mae, rmse, directional_accuracy

print("Evaluation function 'evaluate_model' defined.")

**Reasoning**:
The `evaluate_model` function has been defined in the previous step. Now, I will use this function to evaluate the predictions on both the validation and test datasets and display the calculated MAE, RMSE, and directional accuracy for each.



In [ ]:
# Evaluate the model on the validation set
mae_val, rmse_val, da_val = evaluate_model(y_val, y_pred_val)
print(f"Validation Set Metrics:\n  MAE: {mae_val:.4f}\n  RMSE: {rmse_val:.4f}\n  Directional Accuracy: {da_val:.4f}")

# Evaluate the model on the test set
mae_test, rmse_test, da_test = evaluate_model(y_test, y_pred_test)
print(f"Test Set Metrics:\n  MAE: {mae_test:.4f}\n  RMSE: {rmse_test:.4f}\n  Directional Accuracy: {da_test:.4f}")

## XGBoost Model

XGBoost (eXtreme Gradient Boosting) is a powerful and popular open-source gradient boosting framework known for its speed and performance. It is an ensemble learning method that sequentially builds multiple decision trees, with each new tree correcting the errors of the previous ones. XGBoost is widely used in various machine learning competitions and real-world applications due to its efficiency and accuracy.

### Advantages of XGBoost:

*   **Performance**: XGBoost is highly optimized and designed for efficiency, scalability, and portability. It provides high accuracy and speed, making it a top choice for complex predictive tasks.
*   **Regularization**: It incorporates both L1 (Lasso) and L2 (Ridge) regularization techniques to prevent overfitting. This helps in building a more generalized model that performs well on unseen data.
*   **Handling Missing Values**: XGBoost can effectively handle missing values internally. It learns the best imputation strategy for missing data during the tree-building process.
*   **Flexibility**: The framework is highly flexible, supporting various objective functions (e.g., regression, classification) and custom evaluation metrics, allowing it to be tailored to specific problem requirements.

### Suitability for Stock Return Prediction:

XGBoost is particularly well-suited for stock return prediction due to several factors inherent in financial time series data:

*   **Capturing Complex Non-linear Relationships**: Stock returns are often influenced by intricate and non-linear interactions between various financial indicators. XGBoost's tree-based ensemble approach excels at modeling these complex relationships without requiring explicit feature engineering for non-linearity.
*   **Robustness to Noisy Data**: Financial markets are inherently noisy, with many irrelevant or highly correlated features. XGBoost's regularization techniques and ensemble nature make it robust to noise and less prone to overfitting, which is crucial for generalizing patterns in volatile markets.
*   **Feature Importance**: XGBoost provides insights into feature importance, which can help in understanding which financial indicators are most influential in predicting stock returns, offering interpretability that can be valuable in financial modeling.

### Instantiate and Fit XGBoost Model

### Subtask:
Import `XGBRegressor`, initialize it with default parameters, and fit the model to the `X_train_scaled` and `y_train` datasets.


**Reasoning**:
To initialize and fit an XGBoost model, I will import `XGBRegressor`, instantiate it with default parameters, and then fit it to the `X_train_scaled` and `y_train` datasets.



In [ ]:
from xgboost import XGBRegressor

# Initialize XGBoost Regressor with default parameters
xgb_model = XGBRegressor(n_estimators=500,
                         learning_rate=0.01,
                         max_depth=3,
                         subsample=0.8,
                         colsample_bytree=0.8,
                         early_stopping_rounds=50,
                         random_state=42)

# Fit the model to the scaled training data
xgb_model.fit(X_train_scaled, y_train, eval_set=[(X_val_scaled, y_val)], verbose=100)

print("XGBoost Regressor model initialized and fitted.")

### Make Predictions with XGBoost

### Subtask:
Generate predictions on X_val_scaled and X_test_scaled using the fitted XGBoost model.


**Reasoning**:
I will generate predictions for the validation and test sets using the fitted XGBoost model and store them in new variables.



In [ ]:
y_pred_val_xgb = xgb_model.predict(X_val_scaled)
y_pred_test_xgb = xgb_model.predict(X_test_scaled)

print("Predictions generated for validation and test sets using XGBoost.")

### Evaluate XGBoost Model Performance

### Subtask:
Use the previously defined evaluate_model function to calculate and display Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and Directional Accuracy for the XGBoost model on both the validation and test sets.


**Reasoning**:
I will use the `evaluate_model` function to calculate MAE, RMSE, and Directional Accuracy for the XGBoost model on both the validation and test sets and then print the results, as instructed.



In [ ]:
# Evaluate the XGBoost model on the validation set
mae_val_xgb, rmse_val_xgb, da_val_xgb = evaluate_model(y_val, y_pred_val_xgb)
print(f"XGBoost Validation Set Metrics:\n  MAE: {mae_val_xgb:.4f}\n  RMSE: {rmse_val_xgb:.4f}\n  Directional Accuracy: {da_val_xgb:.4f}")

# Evaluate the XGBoost model on the test set
mae_test_xgb, rmse_test_xgb, da_test_xgb = evaluate_model(y_test, y_pred_test_xgb)
print(f"XGBoost Test Set Metrics:\n  MAE: {mae_test_xgb:.4f}\n  RMSE: {rmse_test_xgb:.4f}\n  Directional Accuracy: {da_test_xgb:.4f}")

## Prepare Data for LSTM

# Task
Define a function to prepare time series data for an LSTM model by creating sequences with a lookback window of 20. Apply this function to `X_train_scaled`, `y_train`, `X_val_scaled`, `y_val`, `X_test_scaled`, and `y_test` to generate `X_train_seq`, `y_train_seq`, `X_val_seq`, `y_val_seq`, `X_test_seq`, and `y_test_seq`. Finally, print the shapes of these newly created sequence datasets.


### Subtask:
Define a function to create sequences (samples, timesteps, features) from the scaled data using a lookback window of 20. Apply this function to `X_train_scaled`, `y_train`, `X_val_scaled`, `y_val`, `X_test_scaled`, and `y_test`.


**Reasoning**:
I need to define the `create_sequences` function as described, then apply it to the scaled feature and target datasets (training, validation, and test sets) using a lookback window of 20, and finally print the shapes of the resulting sequences to verify the data structure. This will prepare the data in the required format for LSTM models.



In [ ]:
def create_sequences(data, lookback):
    X_seq = []
    for i in range(lookback, len(data)):
        X_seq.append(data.iloc[i-lookback : i].values)
    return np.array(X_seq)

lookback_window = 20

# Apply to training data
X_train_seq = create_sequences(X_train_scaled, lookback_window)
y_train_seq = y_train.iloc[lookback_window:].values.reshape(-1, 1)

# Apply to validation data
X_val_seq = create_sequences(X_val_scaled, lookback_window)
y_val_seq = y_val.iloc[lookback_window:].values.reshape(-1, 1)

# Apply to test data
X_test_seq = create_sequences(X_test_scaled, lookback_window)
y_test_seq = y_test.iloc[lookback_window:].values.reshape(-1, 1)

print(f"Shape of X_train_seq: {X_train_seq.shape}")
print(f"Shape of y_train_seq: {y_train_seq.shape}")
print(f"Shape of X_val_seq: {X_val_seq.shape}")
print(f"Shape of y_val_seq: {y_val_seq.shape}")
print(f"Shape of X_test_seq: {X_test_seq.shape}")
print(f"Shape of y_test_seq: {y_test_seq.shape}")

## PyTorch Datasets and DataLoaders

For deep learning models in PyTorch, it's best practice to encapsulate data handling within `Dataset` and `DataLoader` classes. This allows for efficient loading, batching, and shuffling of data.

### Custom TimeSeriesDataset Class

We will define a custom `TimeSeriesDataset` class that inherits from `torch.utils.data.Dataset`. This class will:

1.  **`__init__(self, X, y)`**: Initialize the dataset with our `X_seq` (features) and `y_seq` (targets) data. We'll convert them to PyTorch tensors.
2.  **`__len__(self)`**: Return the total number of samples in the dataset.
3.  **`__getitem__(self, idx)`**: Retrieve a single sample (a feature sequence and its corresponding target) given an index `idx`.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# Check for GPU and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        # Convert data to PyTorch tensors and ensure float32 type, move to device
        self.X = torch.tensor(X, dtype=torch.float32).to(device)
        self.y = torch.tensor(y, dtype=torch.float32).to(device)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

print("TimeSeriesDataset class defined.")

### GPU Acceleration Setup

To maximize the computational efficiency of our deep learning models, particularly the LSTM, this notebook is configured to leverage GPU (Graphics Processing Unit) acceleration when available. PyTorch models and data can be seamlessly moved to a CUDA-enabled GPU for significantly faster training and inference.

**Mechanism:**

1.  **Device Detection**: The line `device = torch.device("cuda" if torch.cuda.is_available() else "cpu")` automatically detects if a CUDA-compatible GPU is present. If so, `device` is set to `'cuda'`; otherwise, it defaults to `'cpu'`.
2.  **Data Transfer**: In the `TimeSeriesDataset` class, both `self.X` (features) and `self.y` (targets) are moved to the `device` during initialization (`.to(device)`). This ensures that all data fed into the model is already residing on the GPU (if available), minimizing CPU-GPU data transfer overhead during training.
3.  **Model Transfer**: The `LSTMModel` instance itself is moved to the `device` after its creation (`.to(device)`). This ensures that all model parameters and computations are performed on the GPU.
4.  **Hidden State Initialization**: The LSTM's initial hidden and cell states (`h0`, `c0`) are explicitly initialized on the same `input_seq.device` within the `forward` method. This prevents the states from being created on the CPU and then transferred, further optimizing GPU utilization.
5.  **Prediction Output Transfer**: When generating predictions, the model's output is moved back to the CPU (`.cpu().numpy()`) before being converted to a NumPy array for compatibility with evaluation metrics, as these typically operate on CPU-bound NumPy arrays.

This setup ensures that if you are running this notebook in an environment with a GPU (like Google Colab's T4 GPU runtime), the intensive computations of the LSTM model will benefit from parallel processing, leading to much quicker execution times. If a GPU is not available, the code gracefully falls back to using the CPU without requiring any manual changes.

### Create Dataset Instances

We now use our custom `TimeSeriesDataset` class to create three separate instances: `train_dataset`, `val_dataset`, and `test_dataset`. Each instance encapsulates the preprocessed `X_seq` features and `y_seq` targets for its respective data split (training, validation, or test).

This step converts our NumPy arrays into PyTorch-compatible `Dataset` objects, which are essential for feeding data into deep learning models.

We will also print the size of each dataset to confirm that the data has been correctly structured and assigned to the datasets.

In [ ]:
# Create Dataset instances
train_dataset = TimeSeriesDataset(X_train_seq, y_train_seq)
val_dataset = TimeSeriesDataset(X_val_seq, y_val_seq)
test_dataset = TimeSeriesDataset(X_test_seq, y_test_seq)

print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

### Create DataLoader Instances

`DataLoader` is a key utility in PyTorch that wraps a `Dataset` and provides an iterable over it. This allows for efficient batch processing of data, which is crucial for training neural networks.

Here's how we configure our `DataLoader` instances:

1.  **`batch_size`**: We define a `batch_size` (e.g., 32). This determines the number of samples processed in one forward/backward pass during training. Using batches makes the training process more stable and memory-efficient.
2.  **`shuffle=True` (for training data)**: For the `train_loader`, we set `shuffle=True` to randomly reorder the data at the beginning of each epoch. This helps prevent the model from learning the order of samples and improves generalization.
3.  **`shuffle=False` (for validation and test data)**: For `val_loader` and `test_loader`, `shuffle` is set to `False` because the order of evaluation does not impact model training, and consistent order can be useful for debugging or comparing predictions.

Finally, an example of one batch is printed to demonstrate the shape of the `X_batch` (features) and `y_batch` (targets) that the LSTM model will receive.

In [ ]:
batch_size = 32 # Define batch size

# Create DataLoader instances
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"DataLoaders created with batch size: {batch_size}")

# Example of one batch
for X_batch, y_batch in train_loader:
    print(f"X_batch shape: {X_batch.shape}")
    print(f"y_batch shape: {y_batch.shape}")
    break

## Define LSTM Model

Now, we'll define the architecture of our Long Short-Term Memory (LSTM) neural network. LSTMs are particularly well-suited for sequence prediction tasks like time series forecasting because they can learn long-term dependencies in data.

### LSTM Model Class (`LSTMModel`)

Our `LSTMModel` will inherit from `torch.nn.Module` and will consist of:

1.  **LSTM Layers**: One or more LSTM layers to process the input sequences. These layers will capture temporal patterns in our stock data.
2.  **Linear Layer**: A final fully connected (linear) layer to map the output of the LSTM layers to a single predicted value (the stock return).

The `__init__` method will set up these layers, and the `forward` method will define how data flows through the network.

In [ ]:
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(LSTMModel, self).__init__()
        self.hidden_layer_size = hidden_size
        self.num_layers = num_layers

        # Apply dropout only if num_layers > 1, otherwise set to 0 to avoid warning
        lstm_dropout = dropout if num_layers > 1 else 0

        self.lstm = nn.LSTM(input_size=input_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            dropout=lstm_dropout
                            )

        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, input_seq):
        # Initialize hidden and cell states on the same device as the input
        h0 = torch.zeros(self.num_layers, input_seq.size(0), self.hidden_layer_size).to(input_seq.device)
        c0 = torch.zeros(self.num_layers, input_seq.size(0), self.hidden_layer_size).to(input_seq.device)

        out, _ = self.lstm(input_seq, (h0, c0))
        out = self.linear(out[:, -1, :])
        return out.view(-1, 1) # Explicitly reshape to (batch_size, 1)

n_features = X_train_seq.shape[2]

print("LSTMModel class defined.")

### LSTM Model Initialization, Training, and Evaluation

This cell performs several critical steps:

1.  **Model Initialization**: An instance of `LSTMModel` is created with specific hyperparameters: `input_size` (number of features), `hidden_size` (neurons in the hidden layer), `num_layers` (depth of the LSTM), and `dropout` (regularization to prevent overfitting).
2.  **Loss Function and Optimizer**: `nn.MSELoss()` (Mean Squared Error) is chosen as the loss function, suitable for regression tasks, measuring the average squared difference between actual and predicted values. The `Adam` optimizer is used, which adapts the learning rate during training, typically leading to faster convergence.
3.  **`run_epoch` Helper Function**: This function encapsulates the logic for a single training or validation epoch. It sets the model to `train()` or `eval()` mode, iterates through batches, performs forward passes, calculates loss, and, for training, executes backpropagation and optimizer steps.
4.  **Training Loop with Early Stopping**: The model is trained over a maximum of 100 epochs. Early stopping is implemented to prevent overfitting and save computational resources. It monitors the validation loss (`val_loss`). If the validation loss does not improve for `patience` (10) consecutive epochs, training stops, and the model's weights are reverted to the state that yielded the best validation loss.
5.  **`get_predictions` Helper Function**: This function puts the model in evaluation mode (`model.eval()`) and uses `torch.no_grad()` to efficiently generate predictions on a given DataLoader without computing gradients.
6.  **Prediction and Evaluation**: Finally, predictions are generated for both the validation (`val_preds`) and test (`test_preds`) sets using the best-performing model. The `evaluate_model` function, previously defined, is then called to calculate and print MAE, RMSE, and Directional Accuracy for both sets, providing a quantitative assessment of the LSTM model's performance.

In [ ]:
# Initialize the LSTM model with defined hyperparameters
model = LSTMModel(
    input_size=n_features,  # Number of input features per time step
    hidden_size=64,         # Number of features in the hidden state
    num_layers=2,           # Number of recurrent layers
    dropout=0.2             # Dropout rate for regularization
).to(device) # Move model to the selected device

# Define the loss function (Mean Squared Error for regression) and optimizer (Adam)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Helper function to run a single epoch (training or validation)
def run_epoch(loader, train=True):
    # Set model to training or evaluation mode
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0
    # Enable or disable gradient calculations based on mode
    with torch.set_grad_enabled(train):
        for X_batch, y_batch in loader:
            # Move data to the selected device (if not already there due to TimeSeriesDataset)
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            # Make predictions
            preds = model(X_batch)
            # Calculate loss
            loss  = criterion(preds, y_batch)
            if train:
                # Backpropagation during training
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(X_batch) # Accumulate total loss

    return total_loss / len(loader.dataset) # Return average loss per sample

# --- Training loop with early stopping ---
best_val_loss = float("inf") # Initialize best validation loss
patience      = 10           # Number of epochs to wait for improvement
wait          = 0            # Counter for epochs without improvement
best_weights  = None         # To store the best model weights

for epoch in range(100): # Maximum number of epochs
    train_loss = run_epoch(train_loader, train=True) # Run training epoch
    val_loss   = run_epoch(val_loader,   train=False) # Run validation epoch

    print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

    # Check for early stopping condition
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights  = model.state_dict().copy() # Save best model weights
        wait          = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

# Restore model to the weights that achieved the best validation loss
model.load_state_dict(best_weights)

# Helper function to get predictions from the model
def get_predictions(loader):
    model.eval() # Set model to evaluation mode
    preds = []
    with torch.no_grad(): # Disable gradient calculations
        for X_batch, _ in loader:
            X_batch = X_batch.to(device) # Move data to device for prediction
            preds.extend(model(X_batch).cpu().numpy()) # Get predictions, move to CPU, convert to numpy
    return np.array(preds)

# Generate predictions for validation and test sets
val_preds  = get_predictions(val_loader)
test_preds = get_predictions(test_loader)

# Evaluate the LSTM model's performance on validation and test sets
mae_val_lstm, rmse_val_lstm, da_val_lstm = evaluate_model(y_val_seq,  val_preds)
print(f"LSTM Validation Set Metrics:\n  MAE: {mae_val_lstm:.4f}\n  RMSE: {rmse_val_lstm:.4f}\n  Directional Accuracy: {da_val_lstm:.4f}")

mae_test_lstm, rmse_test_lstm, da_test_lstm = evaluate_model(y_test_seq, test_preds)
print(f"LSTM Test Set Metrics:\n  MAE: {mae_test_lstm:.4f}\n  RMSE: {rmse_test_lstm:.4f}\n  Directional Accuracy: {da_test_lstm:.4f}")

## 6. Hyperparameter Optimization

### Optuna Objective Function

The `objective` function is the core of Optuna's hyperparameter optimization process. For each `trial` that Optuna conducts, this function defines a complete training and evaluation pipeline. Its goal is to return a metric that Optuna will either maximize or minimize to find the best set of hyperparameters.

Here's a detailed breakdown of what happens within each `trial` in our `objective` function:

1.  **Sample Hyperparameters**: Optuna proposes a unique combination of hyperparameters for each trial. These parameters are sampled from predefined ranges using methods like `trial.suggest_categorical`, `trial.suggest_int`, and `trial.suggest_float`. For our LSTM model, we sample:
    *   `hidden_size`: Number of hidden units in the LSTM layers.
    *   `num_layers`: Number of stacked LSTM layers.
    *   `dropout`: Dropout rate for regularization.
    *   `lr`: Learning rate for the Adam optimizer.
    *   `lookback`: The size of the historical window used to create sequences.
    *   `batch_size`: Number of samples processed per iteration during training.

2.  **Prepare Data**: Crucially, the `lookback` hyperparameter can change for each trial. Therefore, **data sequencing must be rebuilt within the `objective` function** for each trial. New sequences (`X_train_seq_trial`, `y_train_seq_trial`, etc.) are created using the `create_sequences` function with the `lookback` value specific to the current trial. This ensures that the LSTM model is always trained and evaluated on data properly formatted for its chosen lookback window.

3.  **Build and Train Model**: An `LSTMModel` instance is created using the `hidden_size`, `num_layers`, and `dropout` sampled for the current trial. An `Adam` optimizer with the trial's `lr` and an `MSELoss` criterion are also initialized. The model then undergoes a training loop over a maximum of 100 epochs, iterating through `train_loader` batches.

4.  **Early Stopping and Pruning**: An early stopping mechanism is implemented using a `patience` of 10 epochs. If the validation loss (`val_loss`) does not improve for 10 consecutive epochs, the training for that trial is halted. Furthermore, Optuna's `pruner` (in our case, `MedianPruner`) is integrated. It monitors the `val_loss` and can stop unpromising trials early based on their intermediate performance, saving significant computational resources.

5.  **Evaluate Performance**: After the training loop (or early stopping), the model's performance is evaluated on the validation set (`val_loader`). The key metric for optimization in this study is **directional accuracy**. This measures the proportion of times the model correctly predicts the direction of the stock return (up or down). By using `np.sign(val_preds) == np.sign(y_val_seq_trial)`, we calculate this metric.

6.  **Return Metric**: The `objective` function returns the calculated `directional_accuracy`. Optuna is configured to **maximize** this value, guiding the search towards hyperparameters that yield the best directional prediction performance. This approach directly addresses the practical goal of predicting the trend of stock movement, acknowledging that a more complex model (indicated by more layers or larger hidden sizes) does not always lead to a better outcome, particularly in noisy financial data where simpler models might generalize better or capture directional trends more effectively.

In [ ]:
!pip install optuna

In [ ]:
import optuna

### Optuna Objective Function Definition

The following code block defines the `objective` function that Optuna will use for hyperparameter optimization. This function encapsulates the entire training and evaluation process for a single trial, returning the metric to be optimized (directional accuracy in this case).

In [ ]:
def objective(trial):

    # Sample hyperparameters
    hidden_size  = trial.suggest_categorical("hidden_size", [32, 64, 128])
    num_layers   = trial.suggest_int("num_layers", 1, 3)
    dropout      = trial.suggest_float("dropout", 0.1, 0.5)
    lr           = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    lookback     = trial.suggest_categorical("lookback", [10, 20, 30, 60])
    batch_size   = trial.suggest_categorical("batch_size", [16, 32, 64])

    # Rebuild sequences with this trial's lookback for X
    X_train_seq_trial = create_sequences(X_train_scaled, lookback)
    X_val_seq_trial   = create_sequences(X_val_scaled,   lookback)

    # Generate y sequences by slicing, matching how it was done before
    # Ensure y_train and y_val are correctly aligned with the new lookback
    y_train_seq_trial = y_train.iloc[lookback:].values.reshape(-1, 1)
    y_val_seq_trial   = y_val.iloc[lookback:].values.reshape(-1, 1)

    # Check for empty sequences, especially if lookback is too large for a small split
    if len(X_train_seq_trial) == 0 or len(X_val_seq_trial) == 0:
        raise optuna.exceptions.TrialPruned(f"Lookback {lookback} too large for data splits.")

    # DataLoaders
    train_loader = DataLoader(
        TimeSeriesDataset(X_train_seq_trial, y_train_seq_trial),
        batch_size=batch_size, shuffle=True # Shuffle training data
    )
    val_loader = DataLoader(
        TimeSeriesDataset(X_val_seq_trial, y_val_seq_trial),
        batch_size=batch_size, shuffle=False
    )

    # Build model
    model = LSTMModel(
        input_size=X_train_seq_trial.shape[2],
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout
    ).to(device) # Move model to the selected device

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    # Training loop
    best_val_loss = float("inf")
    patience, wait = 10, 0

    for epoch in range(100):
        # Train
        model.train()
        for X_batch, y_batch in train_loader:
            # Move data to the selected device (if not already there due to TimeSeriesDataset)
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch)
            loss  = criterion(preds, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Validate
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                # Move data to the selected device (if not already there due to TimeSeriesDataset)
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                preds    = model(X_batch)
                val_loss += criterion(preds, y_batch).item() * len(X_batch)
        val_loss /= len(val_loader.dataset)

        # Optuna pruning integration
        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    # Evaluate directional accuracy on val set
    val_preds = []
    model.eval()
    with torch.no_grad():
        for X_batch, _ in val_loader:
            X_batch = X_batch.to(device) # Move data to device for prediction
            val_preds.extend(model(X_batch).cpu().numpy()) # Get predictions, move to CPU, convert to numpy

    val_preds    = np.array(val_preds)
    dir_accuracy = np.mean(np.sign(val_preds) == np.sign(y_val_seq_trial))

    return dir_accuracy

## Create an Optuna study and optimize
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=30))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Best trial:")
print(f"  Value: {study.best_value:.4f}")
print("  Params:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

In [ ]:
# Create an Optuna study and optimize
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=30))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Best trial:")
print(f"  Value: {study.best_value:.4f}")
print("  Params:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

## Final Model Training and Evaluation

After Optuna has identified the optimal set of hyperparameters, the final step is to train an LSTM model using these best parameters. This involves:

1.  **Retrieving Best Hyperparameters**: Extracting the `lookback`, `hidden_size`, `num_layers`, `dropout`, `lr`, and `batch_size` from `study.best_params`.
2.  **Rebuilding Sequences**: Re-creating the input sequences (`X_train_seq_final`, `y_train_seq_final`, `X_val_seq_final`, `y_val_seq_final`, `X_test_seq_final`, `y_test_seq_final`) using the best `lookback` window. This ensures all data is prepared correctly for the chosen architecture.
3.  **Combining Training and Validation Data**: To maximize the data available for training the final model, the training and validation sets are concatenated (`X_trainval`, `y_trainval`). This combined dataset is then used to create the `trainval_loader`.
4.  **Initializing Final Model**: An `LSTMModel` instance is created with the best `hidden_size`, `num_layers`, `dropout`, and `input_size`.
5.  **Training the Final Model**: The model is trained on the combined `trainval_loader` using the best `lr` and the `MSELoss` criterion for a fixed number of epochs (100 in this case), without early stopping from the validation set since it's now part of the training data.
6.  **Evaluating on Test Set**: Finally, predictions are generated on the `test_loader`, and the model's performance is evaluated using MAE, RMSE, and Directional Accuracy. This provides an unbiased assessment of how the hyperparameter-tuned LSTM model performs on unseen data.

In [ ]:
best_params = study.best_params

# Rebuild sequences with best lookback
lookback = best_params["lookback"]
X_train_seq_final = create_sequences(X_train_scaled, lookback)
y_train_seq_final = y_train.iloc[lookback:].values.reshape(-1, 1)

X_val_seq_final   = create_sequences(X_val_scaled,   lookback)
y_val_seq_final   = y_val.iloc[lookback:].values.reshape(-1, 1)

X_test_seq_final  = create_sequences(X_test_scaled,  lookback)
y_test_seq_final  = y_test.iloc[lookback:].values.reshape(-1, 1)

# Combine train + val for final training
X_trainval = np.concatenate([X_train_seq_final, X_val_seq_final])
y_trainval = np.concatenate([y_train_seq_final, y_val_seq_final])

trainval_dataset = TimeSeriesDataset(X_trainval, y_trainval)
test_dataset = TimeSeriesDataset(X_test_seq_final, y_test_seq_final)

trainval_loader = DataLoader(
    trainval_dataset,
    batch_size=best_params["batch_size"], shuffle=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=best_params["batch_size"], shuffle=False
)

# Build and train final model
final_model = LSTMModel(
    input_size=X_trainval.shape[2],
    hidden_size=best_params["hidden_size"],
    num_layers=best_params["num_layers"],
    dropout=best_params["dropout"]
).to(device) # Move model to the selected device

optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params["lr"])
criterion = nn.MSELoss()

# Training loop for the final model
print("\nTraining final LSTM model with best hyperparameters...")
for epoch in range(100):
    final_model.train()
    total_train_loss = 0
    for X_batch, y_batch in trainval_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device) # Move data to device
        preds = final_model(X_batch)
        loss  = criterion(preds, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item() * len(X_batch)
    avg_train_loss = total_train_loss / len(trainval_loader.dataset)
    if (epoch + 1) % 10 == 0: # Print loss every 10 epochs
        print(f"Epoch {epoch+1:3d} | Train Loss: {avg_train_loss:.6f}")

# Evaluate on test set
def get_predictions(loader, model):
    model.eval() # Set model to evaluation mode
    preds = []
    with torch.no_grad(): # Disable gradient calculations
        for X_batch, _ in loader:
            X_batch = X_batch.to(device) # Move data to device for prediction
            preds.extend(model(X_batch).cpu().numpy()) # Get predictions, move to CPU, convert to numpy
    return np.array(preds)


test_preds_final = get_predictions(test_loader, final_model)

mae_test_final, rmse_test_final, da_test_final = evaluate_model(y_test_seq_final, test_preds_final)
print(f"\nFinal Tuned LSTM Test Set Metrics:\n  MAE: {mae_test_final:.4f}\n  RMSE: {rmse_test_final:.4f}\n  Directional Accuracy: {da_test_final:.4f}")

## Model Performance Comparison

This section compares the performance of the three models: Ridge Regression, XGBoost, and LSTM, across various metrics including Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and Directional Accuracy. This comparison helps in understanding which model performs best under different criteria and provides insights into the trade-offs between model complexity and predictive power for stock return prediction.

In [ ]:

model_comparison = pd.DataFrame({
    'Model': ['Ridge Regression', 'XGBoost', 'LSTM'],
    'Test RMSE': [rmse_test, rmse_test_xgb, rmse_test_final],
    'Test Directional Accuracy': [da_test, da_test_xgb, da_test_final]
})

display(model_comparison.round(4))

## Sharpe Ratio Calculation

**Reasoning**:
The Sharpe Ratio is a measure of risk-adjusted return, indicating the average return earned in excess of the risk-free rate per unit of volatility or total risk. It is a crucial metric for evaluating the performance of trading strategies, as it considers both returns and the risk taken to achieve them. A higher Sharpe ratio signifies a better risk-adjusted performance.

We will define a function to calculate the Sharpe Ratio, then apply it to the returns generated by each model's trading strategy.

In [ ]:
import numpy as np

def calculate_sharpe_ratio(strategy_returns, risk_free_rate=0.0):
    # Calculate daily excess returns
    excess_returns = strategy_returns - risk_free_rate

    # Calculate the mean of excess returns
    mean_excess_returns = excess_returns.mean()

    # Calculate the standard deviation of excess returns
    std_excess_returns = excess_returns.std()

    # Calculate Sharpe Ratio
    # Annualize if data is daily (sqrt(252) for daily data)
    if std_excess_returns == 0 or np.isnan(std_excess_returns):
        return 0 # Avoid division by zero or NaN
    return (mean_excess_returns / std_excess_returns) * np.sqrt(252)

# Assuming a risk-free rate of 0 for simplicity (can be adjusted if known)
RISK_FREE_RATE = 0.0

# Get strategy returns for Ridge (daily percentage change)
ridge_signals = (pd.Series(y_pred_test, index=y_test.index) > 0).astype(int)
ridge_daily_returns = y_test * ridge_signals
sharpe_ridge = calculate_sharpe_ratio(ridge_daily_returns, RISK_FREE_RATE)

# Get strategy returns for XGBoost
xgb_signals = (pd.Series(y_pred_test_xgb, index=y_test.index) > 0).astype(int)
xgb_daily_returns = y_test * xgb_signals
sharpe_xgb = calculate_sharpe_ratio(xgb_daily_returns, RISK_FREE_RATE)

# Get strategy returns for LSTM
# Define y_true_lstm_slice and y_pred_lstm_series for Sharpe Ratio calculation
lstm_actual_y_slice = y_test.iloc[best_params["lookback"]:]
y_pred_lstm_series = pd.Series(test_preds_final.flatten(), index=lstm_actual_y_slice.index)
lstm_signals = (y_pred_lstm_series > 0).astype(int)
lstm_daily_returns = lstm_actual_y_slice * lstm_signals
sharpe_lstm = calculate_sharpe_ratio(lstm_daily_returns, RISK_FREE_RATE)

print(f"Ridge Strategy Sharpe Ratio: {sharpe_ridge:.4f}")
print(f"XGBoost Strategy Sharpe Ratio: {sharpe_xgb:.4f}")
print(f"LSTM Strategy Sharpe Ratio: {sharpe_lstm:.4f}")

## 7. Backtesting

For **Backtesting**, we simulate simple trading strategies based on the directional predictions of each model and evaluate their cumulative returns. This provides a more practical assessment of model performance in a real-world trading scenario.

### Strategy:
We employ a simple long-only strategy:
*   If the model predicts a positive log return for the next day, we 'buy' (go long) and realize the actual log return for that day.
*   If the model predicts a non-positive (zero or negative) log return, we 'hold cash' (do not trade) and realize zero return for that day.

We will compare the cumulative returns of these strategies against a simple 'buy and hold' strategy.

## Model Visualizations

To further understand the models' performance, we visualize the predicted vs. actual log returns and the cumulative returns of simple trading strategies based on each model's predictions.

In [ ]:
# Create a DataFrame for actual vs. predicted values for the test set
# Using XGBoost predictions as it had the highest directional accuracy
plot_df_xgb = pd.DataFrame({
    'Actual': y_test,
    'XGBoost Predicted': y_pred_test_xgb
}, index=y_test.index)

plt.figure(figsize=(14, 7))
plt.plot(plot_df_xgb.index, plot_df_xgb['Actual'], label='Actual Log Return', alpha=0.7)
plt.plot(plot_df_xgb.index, plot_df_xgb['XGBoost Predicted'], label='XGBoost Predicted Log Return', alpha=0.7)
plt.title('Actual vs. XGBoost Predicted Log Returns on Test Set')
plt.xlabel('Date')
plt.ylabel('Log Return')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Helper function to calculate cumulative returns based on directional predictions
def calculate_cumulative_returns(y_true_series, y_pred_series):
    # Ensure both are Series/Arrays and have aligned indices
    # For simplicity, we'll consider a long-only strategy:
    # if predicted return > 0, we take the actual return, else 0 (hold cash)

    # Convert predictions to directional signals (1 for positive, 0 for non-positive)
    signals = (y_pred_series > 0).astype(int)
    strategy_returns = y_true_series * signals

    # Calculate cumulative returns: (1 + r1) * (1 + r2) * ...
    cumulative_returns = (1 + strategy_returns).cumprod()
    return cumulative_returns

# Calculate cumulative returns for Ridge Regression
cumulative_ridge = calculate_cumulative_returns(y_test, pd.Series(y_pred_test, index=y_test.index))

# Calculate cumulative returns for XGBoost
cumulative_xgb = calculate_cumulative_returns(y_test, pd.Series(y_pred_test_xgb, index=y_test.index))

# Calculate cumulative returns for LSTM
# Need to align y_test with LSTM's predictions based on the lookback window
lstm_actual_y_slice = y_test.iloc[best_params["lookback"]:]
y_pred_lstm_series = pd.Series(test_preds_final.flatten(), index=lstm_actual_y_slice.index)
cumulative_lstm = calculate_cumulative_returns(lstm_actual_y_slice, y_pred_lstm_series)

# Also calculate cumulative returns for a 'buy and hold' strategy as a baseline
buy_hold_returns = (np.exp(y_test)).cumprod() # Corrected calculation for log returns

# Plotting cumulative returns
plt.figure(figsize=(14, 7))
plt.plot(buy_hold_returns.index, buy_hold_returns, label='Actual Cumulative Return', alpha=0.9, linewidth=2)
plt.plot(cumulative_ridge.index, cumulative_ridge, label='Ridge Strategy Cumulative Return', alpha=0.8)
plt.plot(cumulative_xgb.index, cumulative_xgb, label='XGBoost Strategy Cumulative Return', alpha=0.8)
plt.plot(cumulative_lstm.index, cumulative_lstm, label='LSTM Strategy Cumulative Return', alpha=0.8)

plt.title('Cumulative Returns of Trading Strategies on Test Set')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Insights from the Notebook

Based on the analysis, here are some key insights:

1.  **Feature Engineering Impact**: Creating lag features, rolling statistics, and volume-related features was crucial for providing the models with historical context and market sentiment indicators. These features enriched the dataset, enabling the models to capture complex temporal dependencies.

2.  **Data Scaling Importance**: Scaling features using `StandardScaler` was essential, particularly for the LSTM model, to ensure that features contribute equally to the model's training and to prevent issues caused by differing scales.

3.  **Model Performance Variation**: While all models showed reasonable performance, particularly in terms of MAE and RMSE, there were interesting differences in their directional accuracy.

    *   **Ridge Regression (Baseline)**: Performed surprisingly well as a baseline, indicating that linear relationships and basic lagged features hold significant predictive power for stock returns. Its directional accuracy was competitive, suggesting that simpler models can sometimes capture market direction effectively.

    *   **XGBoost**: Generally showed improved MAE and RMSE compared to Ridge Regression, demonstrating its capability to capture more complex, non-linear patterns. Notably, XGBoost achieved the highest directional accuracy on the test set, suggesting it was best at predicting the *direction* of the stock price movement.

    *   **LSTM**: While LSTMs are theoretically well-suited for sequence data, its performance in terms of directional accuracy was not significantly better than XGBoost or even the Ridge Regression baseline. This observation highlights a critical point: **more complex models do not automatically imply better outputs, especially in noisy financial time series data.** The inherent volatility and unpredictability of stock markets can make it challenging for even advanced neural networks to consistently outperform simpler models on directional prediction tasks. The slight improvement in MAE and RMSE for LSTM over Ridge might indicate better magnitude prediction, but not necessarily better directional calls.

4.  **Directional Accuracy as a Key Metric**: For stock return prediction, directional accuracy is often more critical than solely minimizing MAE or RMSE. A model that consistently predicts the correct direction of movement can be highly valuable, even if the predicted magnitude is not perfectly accurate.

5.  **Hyperparameter Tuning (Optuna)**: The use of Optuna for hyperparameter optimization for the LSTM model was essential. It allowed for the exploration of a wide range of hyperparameters, leading to the selection of the best configuration for the LSTM. However, as noted above, even with tuning, the LSTM's complexity didn't guarantee superior directional accuracy.

In conclusion, while advanced models like XGBoost and LSTM offer powerful capabilities for capturing intricate patterns, the results suggest that for stock return prediction, model complexity does not always directly translate to superior performance, particularly in directional accuracy. Simpler models can provide strong baselines, and careful evaluation across multiple metrics is crucial for selecting the most appropriate model for a given financial prediction task.

## 8. Interpretability

### Ridge Regression Coefficients

**Reasoning**:
Ridge Regression provides coefficients for each feature, indicating their weight or importance in predicting the target variable. By examining these coefficients, we can understand which features the model considers most influential. A higher absolute value of a coefficient suggests a stronger impact, while the sign indicates the direction of that impact (positive for increasing log return, negative for decreasing).

In [ ]:
ridge_coefficients = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'Coefficient': ridge_model.coef_.flatten()
}).sort_values(by='Coefficient', ascending=False)

print("Ridge Regression Coefficients:")
display(ridge_coefficients)

### XGBoost Feature Importance

**Reasoning**:
XGBoost provides a measure of feature importance, which quantifies the contribution of each feature to the model's predictions. This is typically calculated based on the number of times a feature is used to split the data across all trees, or the gain associated with those splits. Analyzing these importances helps us understand which market indicators the XGBoost model considers most critical for predicting stock returns.

In [ ]:
xgb_feature_importance = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("XGBoost Feature Importances:")
display(xgb_feature_importance)

## 9. Limitations

This analysis provides valuable insights into stock return prediction, but it's important to acknowledge its limitations:

1.  **Single Stock Focus**: The models were trained and evaluated exclusively on Apple (AAPL) stock data. The generalizability of these findings to other stocks, industries, or broader market indices is not guaranteed and would require further investigation.
2.  **Limited Feature Set**: The features used were primarily based on historical price and volume data. The stock market is influenced by a vast array of factors, including macroeconomic indicators (e.g., interest rates, inflation, GDP), company-specific news, earnings reports, geopolitical events, and market sentiment. These external factors were not incorporated into the current feature set.
3.  **Simplified Trading Strategy**: The backtesting strategy employed is highly simplified (long-only, hold cash on non-positive predictions). It does not account for real-world trading complexities such as transaction costs, slippage, bid-ask spreads, market liquidity constraints, or advanced risk management techniques (e.g., stop-loss orders, position sizing).
4.  **Noisy Data Challenge**: Financial time series data is inherently noisy and highly stochastic. Even advanced models struggle to capture all the intricacies and non-linearities, leading to limitations in predictive power, especially for precise price movements.
5.  **Model Complexity vs. Performance**: The study highlighted that a more complex model (LSTM) did not necessarily translate to superior directional accuracy compared to simpler models like XGBoost, particularly in a noisy environment. This underscores the challenge of finding the right balance between model complexity and robustness.
6.  **Stationarity Assumptions**: While log returns mitigate some non-stationarity issues, some models might still implicitly rely on assumptions about the underlying data distribution that are not perfectly met in real-world financial data.

## 10. Next Steps

To build upon this analysis and improve the predictive capabilities and robustness of the models, several next steps can be pursued:

1.  **Enrich Feature Set**: Incorporate additional data sources and features that could influence stock returns. This includes:
    *   **Macroeconomic Indicators**: GDP growth, interest rates, inflation, unemployment rates.
    *   **Fundamental Data**: Company earnings, revenue, P/E ratios, debt-to-equity.
    *   **Alternative Data**: News sentiment analysis, social media trends, Google Trends data for the company or sector.
    *   **Technical Indicators**: Explore a wider range of technical indicators beyond simple moving averages and standard deviations (e.g., RSI, MACD, Bollinger Bands).
2.  **Explore Advanced Model Architectures**: Investigate other state-of-the-art models for time series forecasting:
    *   **Transformer Networks**: Models like the Transformer, which excel in sequence-to-sequence tasks, could capture longer-range dependencies.
    *   **Temporal Fusion Transformers (TFTs)**: Designed for multi-horizon forecasting with interpretability.
    *   **Reinforcement Learning**: Frame the trading problem as a reinforcement learning task, where an agent learns to make trading decisions to maximize returns while managing risk.
3.  **Ensemble Modeling**: Combine the predictions from multiple models (e.g., a weighted average of Ridge, XGBoost, and LSTM predictions) to leverage the strengths of each model and potentially achieve more stable and accurate forecasts.
4.  **Robust Backtesting Framework**: Develop a more comprehensive backtesting environment that accounts for:
    *   **Transaction Costs**: Brokerage fees, slippage.
    *   **Risk Management**: Stop-loss and take-profit mechanisms, portfolio diversification, value-at-risk (VaR) calculations.
    *   **Portfolio Optimization**: Develop strategies for allocating capital across multiple assets based on model predictions.
5.  **Multi-Asset Prediction**: Extend the analysis to predict returns for a basket of stocks or an entire index to assess portfolio-level performance and diversification benefits.
6.  **Interpretability Deep Dive**: For models like XGBoost and LSTM, apply advanced interpretability techniques (e.g., SHAP values, LIME) to understand which features drive specific predictions and gain deeper insights into the models' decision-making processes.
7.  **Dynamic Hyperparameter Tuning**: Implement more dynamic hyperparameter tuning strategies, potentially including online learning or adaptive methods that adjust hyperparameters over time as market conditions change.
8.  **Event-Driven Analysis**: Investigate the models' performance around significant market events (e.g., earnings announcements, economic reports) to understand their predictive capabilities during periods of high volatility.